# Run this first !
Ensures correct libraries and imports data

In [1]:

# Run this first !
# project setup — run first, do not edit except NAME
NAME = "rachel" # <<< your name

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
               check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}  data {DATA}")
print(loading.exports(DATA))
print(loading.files(DATA))

utt = loading.load(DATA, "all_utterances.csv", "2026-09-13")
sent = loading.load(DATA, "all_sentences.csv", "2026-09-13")
met = loading.load(DATA, "all_metrics.csv", "2026-09-13")

print(utt.shape, sent.shape, met.shape)
sent.head()
data=sent.copy()

ok  python 3.12.13  pandas 2.2.3  data /Users/rdrobinson/Cambridge/boe-financial-analysis/data
['2026-09-13']
['all_metrics.csv', 'all_sentences.csv', 'all_utterances.csv']
loaded all_utterances.csv  1201 rows
loaded all_sentences.csv  9519 rows
loaded all_metrics.csv  510 rows
(1201, 11) (9519, 12) (510, 6)


In [2]:
print(loading.exports(DATA))
print(loading.files(DATA))

utt = loading.load(DATA, "all_utterances.csv", "2026-09-13")
sent = loading.load(DATA, "all_sentences.csv", "2026-09-13")
met = loading.load(DATA, "all_metrics.csv", "2026-09-13")

print(utt.shape, sent.shape, met.shape)
sent.head()

['2026-09-13']
['all_metrics.csv', 'all_sentences.csv', 'all_utterances.csv']
loaded all_utterances.csv  1201 rows
loaded all_sentences.csv  9519 rows
loaded all_metrics.csv  510 rows
(1201, 11) (9519, 12) (510, 6)


,bank,quarter,call_type,call_date,position_in_call,section,speaker_role,speaker_name,speaker_institution,sentence_id,sentence_number,sentence
0,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,9789,1,"Good morning, ladies and gentlemen."
1,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,9790,2,Welcome to JPMorgan Chase’s First Quarter 2023...
2,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,9791,3,This call is being recorded.
3,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,9792,4,Your line will be muted for the duration of th...
4,JPM,2023-Q1,earnings,2023-04-14,1,prepared,operator,Operator,NaN,9793,5,We will now go live to the presentation.


In [3]:
data=sent.copy()

# Sentence pre-processing (stop words etc)
This is just a starting point as there are several oddities - for example greeting words mean the Bert Emotion skews towards "Joy" and some stopwords such as "us" seems to be treated as a plural of "u" - hence needs removing. Also need to work out how to differentiate between "credit" and "Credit Suisse" as both are valid but cannot be treated as the same

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rdrobinson/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rdrobinson/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/rdrobinson/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rdrobinson/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
def preprocess_text(text):
    if not isinstance(text, str): # Handle non-string input, e.g., NaN
        return []
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize the text
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize
    stop_words = set(stopwords.words('english'))

    # Remove words that are not informative for topic modeling
    stop_words.update(['also', 'quarter', 'year', 'u', 'think', 'thats'])

    greeting_words = {
    "hello", "hi", "dear", "greetings",
    "thank", "thanks", "good", "morning",
    "afternoon", "evening",
    "ladies", "gentlemen", "sir", "madam",
    "yeah", "welcome"
}

    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        if word not in stop_words and word.isalpha() and word not in greeting_words # Ensure only alphabetic words are kept and remove greeting words
    ]
    return [word for word in cleaned_tokens if word not in stop_words]

In [6]:
# Apply the preprocessing function
data['cleaned_text'] = data['sentence'].apply(preprocess_text)

# Display the first few rows
print(data[['sentence', 'cleaned_text']].head())

                                            sentence  \
0                Good morning, ladies and gentlemen.   
1  Welcome to JPMorgan Chase’s First Quarter 2023...   
2                       This call is being recorded.   
3  Your line will be muted for the duration of th...   
4           We will now go live to the presentation.   

                               cleaned_text  
0                                        []  
1  [jpmorgan, chase, first, earnings, call]  
2                          [call, recorded]  
3             [line, muted, duration, call]  
4                  [go, live, presentation]  


# UBS Data

In [7]:
# Create dataframe with bank = ubs
ubs_data = data[data['bank'] == 'UBS'].copy()
print(ubs_data.shape)
ubs_data.head(2)

(4714, 13)


,bank,quarter,call_type,call_date,position_in_call,section,speaker_role,speaker_name,speaker_institution,sentence_id,sentence_number,sentence,cleaned_text
4805,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5075,1,"Thank you, Sarah, good morning, everyone.","[sarah, everyone]"
4806,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5076,2,I am happy to be back here with all of you.,"[happy, back]"


Now need to split this further into prepared (from the presentation) and q&a

In [8]:
# Create ubs dataframe with section = 'prepared'
ubs_prepared_data = ubs_data[ubs_data['section'] == 'prepared'].copy()
print(ubs_prepared_data.shape)
ubs_prepared_data.head(2)


(2061, 13)


,bank,quarter,call_type,call_date,position_in_call,section,speaker_role,speaker_name,speaker_institution,sentence_id,sentence_number,sentence,cleaned_text
4805,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5075,1,"Thank you, Sarah, good morning, everyone.","[sarah, everyone]"
4806,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5076,2,I am happy to be back here with all of you.,"[happy, back]"


In [9]:
# Count different numbers of speakers in UBS prepared section
ubs_prepared_speaker_counts = ubs_prepared_data['speaker_name'].value_counts()
print(ubs_prepared_speaker_counts)

speaker_name
Todd Tuckner         1221
Sergio P. Ermotti     752
Sarah Youngwood        88
Name: count, dtype: int64


In [10]:
# Create ubs dataframe with section = 'q&a'
ubs_qa_data = ubs_data[ubs_data['section'] == 'qa'].copy()
print(ubs_qa_data.shape)
ubs_qa_data.head(2)

(2653, 13)


,bank,quarter,call_type,call_date,position_in_call,section,speaker_role,speaker_name,speaker_institution,sentence_id,sentence_number,sentence,cleaned_text
4983,UBS,2023-Q1,earnings,2023-04-25,3,qa,analyst,Chris Hallam,Goldman Sachs,5253,1,Yes.,[yes]
4984,UBS,2023-Q1,earnings,2023-04-25,3,qa,analyst,Chris Hallam,Goldman Sachs,5254,2,"Good morning, everybody.",[everybody]


In [11]:
# Count different numbers of speakers in UBS q&a section
ubs_qa_speaker_counts = ubs_qa_data['speaker_name'].value_counts()
print(ubs_qa_speaker_counts)

speaker_name
Todd Tuckner         807
Sergio P. Ermotti    628
Andrew Coombs        128
Jeremy Sigee         112
Kian Abouhossein     102
Chris Hallam          90
Giulia Miotto         83
Amit Goel             79
Benjamin Goy          76
Anke Reingen          74
Stefan Stalmann       69
Adam Terelak          61
Sarah Youngwood       61
Andrew Lim            60
Alastair Ryan         48
Piers Brown           45
Tom Hallett           40
Flora Bocahut         36
Nicolas Payen         21
Antonio Reale         17
Vishal Shah           11
Sarah Mackey           5
Name: count, dtype: int64


## Bert Emotion: UBS Presentation section
Truncation = True means only 512 tokens (350 -400 words) are kept. Anything longer is ignored. Also output contains top_k = None so that the scores for all the emotions are retained to allow comparison later on

In [12]:
# Conduct Bert Emotion analysis on cleaned sentences from presentation section (JPM)
from transformers import pipeline
emotion_analyzer = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion")
documents = (
    ubs_prepared_data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )

/Users/rdrobinson/Cambridge/boe-financial-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7554.12it/s]


In [14]:
# Run the model and keep all emotion scores
basic_emotion = emotion_analyzer(documents, top_k=None, truncation=True)

# Flatten the model output into one dict per sentence
emotion_rows = []
for result in basic_emotion:
    if isinstance(result, list):
        emotion_rows.append({entry["label"]: entry["score"] for entry in result})
    elif isinstance(result, dict):
        emotion_rows.append({result["label"]: result["score"]})
    else:
        emotion_rows.append({})

# Add the top emotion and score to the dataframe
ubs_prepared_data["main_emotion"] = [
    max(row, key=row.get, default="neutral") if row else "neutral"
    for row in emotion_rows
]
ubs_prepared_data["main_emotion_score"] = [
    row.get(max(row, key=row.get, default="neutral"), 0.0) if row else 0.0
    for row in emotion_rows
]

# Add all emotion columns as separate numeric columns
for emotion in ["anger", "joy", "fear", "sadness", "love", "surprise"]:
    ubs_prepared_data[emotion] = [row.get(emotion, 0.0) for row in emotion_rows]

In [15]:
ubs_prepared_data.head(5)

,bank,quarter,call_type,call_date,position_in_call,section,speaker_role,speaker_name,speaker_institution,sentence_id,...,sentence,cleaned_text,main_emotion,main_emotion_score,anger,joy,fear,sadness,love,surprise
4805,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5075,...,"Thank you, Sarah, good morning, everyone.","[sarah, everyone]",anger,0.732509,0.732509,0.131366,0.098852,0.026312,0.006042,0.004918
4806,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5076,...,I am happy to be back here with all of you.,"[happy, back]",joy,0.995883,0.000551,0.995883,0.000375,0.001802,0.001019,0.000370
4807,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5077,...,And it is an honor and privilege to lead UBS o...,"[honor, privilege, lead, ubs, especially, pivo...",joy,0.990937,0.002844,0.990937,0.001494,0.003068,0.000985,0.000672
4808,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5078,...,"First of all, I’d like to thank Ralph, the man...","[first, id, like, ralph, management, team, emp...",joy,0.874219,0.075282,0.874219,0.000714,0.034093,0.013854,0.001837
4809,UBS,2023-Q1,earnings,2023-04-25,1,prepared,management,Sergio P. Ermotti,NaN,5079,...,"During this time, UBS delivered record results...","[time, ubs, delivered, record, result, continu...",joy,0.996476,0.000659,0.996476,0.000505,0.001307,0.000728,0.000325


## stopped here

In [16]:
# Plot mean and standard deviation of every emotion every quarter as a scatter plot on the same plot (UBS)
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 8))
for emotion in ubs_prepared_emotion_stats["emotion"].unique():
    emotion_prepared_data = ubs_prepared_emotion_stats[ubs_prepared_emotion_stats["emotion"] == emotion]
    # Add lines
    # use pre-defined colours for emotions
    #color = emotion_colors.get(emotion, "black")
    plt.plot(emotion_prepared_data["quarter"], emotion_prepared_data["mean_score"], label=f"{emotion} (average)", marker='o')
    # Add error bars for standard deviation
    #plt.errorbar(emotion_prepared_data["quarter"], emotion_prepared_data["mean_score"], yerr=emotion_prepared_data["std_score"], fmt='o', alpha=0.5)
plt.title("UBS Presentation Section - Average Emotion Scores Every Quarter")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.legend()
plt.show()



NameError: name 'ubs_prepared_emotion_stats' is not defined

<Figure size 1200x800 with 0 Axes>

In [ ]:
# Plot emotions fear and surprise only as grouped barchart (UBS)
import seaborn as sns
plt.figure(figsize=(12, 8))
fear_surprise_prepared_data = ubs_prepared_emotion_stats[ubs_prepared_emotion_stats["emotion"].isin(["fear", "surprise"])]
sns.barplot(data=fear_surprise_prepared_data, x="quarter", y="mean_score", hue="emotion")
plt.title("UBS Prepared Section - Average Emotion Scores Every Quarter for Prepared Section (Fear and Surprise)")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.show()

## Bert Emotion: UBS Q&A section

In [ ]:
# Conduct Bert Emotion analysis on cleaned sentences from Q&Asection (UBS)
from transformers import pipeline
emotion_analyzer = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion")
documents = (
    ubs_qa_data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )
basic_emotion = emotion_analyzer(documents, truncation=True)

print("Sentence emotion analysis:")
print(basic_emotion)

# Add emotion and score to the dataframe
ubs_qa_data["emotion"] = [item["label"] for item in basic_emotion]
ubs_qa_data["score"] = [item["score"] for item in basic_emotion]
ubs_qa_data.head(2)

In [ ]:
# Create dataframe to calculate mean and standard deviation of every emotion every quarter (UBS)
ubs_qa_emotion_stats = ubs_qa_data.groupby(["quarter", "emotion"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std")
).reset_index()
ubs_qa_emotion_stats.head(10)

In [ ]:
# Plot mean and standard deviation of every emotion every quarter as a scatter plot on the same plot (UBS)
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 8))
for emotion in ubs_qa_emotion_stats["emotion"].unique():
    emotion_qa_data = ubs_qa_emotion_stats[ubs_qa_emotion_stats["emotion"] == emotion]
    # Add lines
    # use pre-defined colours for emotions
    #color = emotion_colors.get(emotion, "black")
    plt.plot(emotion_qa_data["quarter"], emotion_qa_data["mean_score"], label=f"{emotion} (average)", marker='o')
plt.title("UBS Q&A Section - Average Emotion Scores Every Quarter")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
# Plot emotions fear and surprise only as grouped barchart (UBS)
import seaborn as sns
plt.figure(figsize=(12, 8))
fear_surprise_qa_data = ubs_qa_emotion_stats[ubs_qa_emotion_stats["emotion"].isin(["fear", "surprise"])]
sns.barplot(data=fear_surprise_qa_data, x="quarter", y="mean_score", hue="emotion")
plt.title("UBS Q&A Section - Average Emotion Scores Every Quarter for Q&A Section (Fear and Surprise)")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.show()

## Main Presenters (UBS)
Create a subset with just the main speakers from the presentation section

In [ ]:
# Filter UBS qa data to just include speaker_name = Todd Tuckner and Sergio P. Ermotti
ubs_qa_filtered = ubs_qa_data[(ubs_qa_data['speaker_name'] == 'Todd Tuckner') | (ubs_qa_data['speaker_name'] == 'Sergio P. Ermotti')]
ubs_qa_filtered.head(2)


In [ ]:
# Create dataframe to calculate mean and standard deviation of every emotion every quarter (UBS)
ubs_qa_filtered_emotion_stats = ubs_qa_filtered.groupby(["quarter", "emotion"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std")
).reset_index()
ubs_qa_filtered_emotion_stats.head(10)

In [ ]:
# Plot emotions fear and surprise only as grouped barchart (UBS)
import seaborn as sns
plt.figure(figsize=(12, 8))
ubs_fear_surprise_qa_filtered_data = ubs_qa_emotion_stats[ubs_qa_emotion_stats["emotion"].isin(["fear", "surprise"])]
sns.barplot(data=ubs_fear_surprise_qa_filtered_data, x="quarter", y="mean_score", hue="emotion")
plt.title("UBS Q&A Section (Todd Tuckner & Sergio P. Ermotti) - Average Emotion Scores Every Quarter for Q&A Section (Fear and Surprise)")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.show()

## Sentiment comparisons - UBS

In [ ]:
#ubs_qa_filtered_emotion_stats
#ubs_prepared_emotion_stats

# Create one dataframe from ubs_prepared_emotion_stats and ubs_qa_filtered_emotion_stats for comparison plots and retain column for prepared vs Q&A
ubs_prepared_emotion_stats["source"] = "prepared"
ubs_qa_filtered_emotion_stats["source"] = "qa_filtered"
ubs_combined_emotion_stats = pd.concat([ubs_prepared_emotion_stats, ubs_qa_filtered_emotion_stats], ignore_index=True)
ubs_combined_emotion_stats.tail(20)

In [ ]:
# subtract prepared mean_score from Q&A mean_score for each emotion and quarter
ubs_diff_emotion_stats = ubs_combined_emotion_stats.pivot_table(index=["quarter", "emotion"], columns="source", values="mean_score").reset_index()
ubs_diff_emotion_stats["diff"] = ubs_diff_emotion_stats["qa_filtered"] - ubs_diff_emotion_stats["prepared"]

ubs_diff_emotion_stats.tail(20)

In [ ]:
# Display the combined emotion stats for UBS Q&A and prepared data as a scatter plot



plt.figure(figsize=(10, 6))
sns.barplot(data=ubs_diff_emotion_stats, x="quarter", y="diff", hue="emotion")
plt.axhline(0, linestyle='--', color='gray')
# Add axis line for 20% change in emotion scores
plt.axhline(0.2, linestyle='--', color='red')
plt.axhline(-0.2, linestyle='--', color='red')
plt.title("UBS: Difference in Emotion Scores for Todd Tuckner & Sergio P. Ermotti (Subtracting Prepared from Q&A)")
plt.show()

# JPM Data

In [ ]:
# Create dataframe with bank = jpm
jpm_data = data[data['bank'] == 'JPM'].copy()
print(jpm_data.shape)
jpm_data.head(2)

In [ ]:
# Create jpm dataframe with section = 'prepared'
jpm_prepared_data = jpm_data[jpm_data['section'] == 'prepared'].copy()
print(jpm_prepared_data.shape)
jpm_prepared_data.head(2)

In [ ]:
# Count different numbers of speakers in JPM prepared section
jpm_prepared_speaker_counts = jpm_prepared_data['speaker_name'].value_counts()
print(jpm_prepared_speaker_counts)

In [ ]:
# Create jpm dataframe with section = 'q&a'
jpm_qa_data = jpm_data[jpm_data['section'] == 'qa'].copy()
print(jpm_qa_data.shape)
jpm_qa_data.head(2)

In [ ]:
# Count different numbers of speakers in JPM q&a section
jpm_qa_speaker_counts = jpm_qa_data['speaker_name'].value_counts()
print(jpm_qa_speaker_counts)


## Bert Emotion: JPM Presentation section

In [ ]:
# Conduct Bert Emotion analysis on cleaned sentences from presentation section (JPM)
from transformers import pipeline
emotion_analyzer = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion")
documents = (
    jpm_prepared_data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )
basic_emotion = emotion_analyzer(documents, truncation=True)

print("Sentence emotion analysis:")
print(basic_emotion)

# Add emotion and score to the dataframe
jpm_prepared_data["emotion"] = [item["label"] for item in basic_emotion]
jpm_prepared_data["score"] = [item["score"] for item in basic_emotion]
jpm_prepared_data.head(2)

In [ ]:
# Create dataframe to calculate mean and standard deviation of every emotion every quarter (UBS)
jpm_prepared_emotion_stats = jpm_prepared_data.groupby(["quarter", "emotion"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std")
).reset_index()
jpm_prepared_emotion_stats.head(10)

In [ ]:
# Plot mean and standard deviation of every emotion every quarter as a scatter plot on the same plot (JPM)
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 8))
for emotion in jpm_prepared_emotion_stats["emotion"].unique():
    jpm_emotion_prepared_data = jpm_prepared_emotion_stats[jpm_prepared_emotion_stats["emotion"] == emotion]
    # Add lines
    # use pre-defined colours for emotions
    #color = emotion_colors.get(emotion, "black")
    plt.plot(jpm_emotion_prepared_data["quarter"], jpm_emotion_prepared_data["mean_score"], label=f"{emotion} (average)", marker='o')
    # Add error bars for standard deviation
    #plt.errorbar(jpm_emotion_prepared_data["quarter"], jpm_emotion_prepared_data["mean_score"], yerr=jpm_emotion_prepared_data["std_score"], fmt='o', alpha=0.5)
plt.title("JPM Presentation Section - Average Emotion Scores Every Quarter")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
# Plot emotions fear and surprise only as grouped barchart (JPM)
import seaborn as sns
plt.figure(figsize=(12, 8))
fear_surprise_prepared_data = jpm_prepared_emotion_stats[jpm_prepared_emotion_stats["emotion"].isin(["fear", "surprise"])]
sns.barplot(data=fear_surprise_prepared_data, x="quarter", y="mean_score", hue="emotion")
plt.title("JPM Prepared Section - Average Emotion Scores Every Quarter for Prepared Section (Fear and Surprise)")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.show()

## Bert Emotion: JPM Q&A section

In [ ]:
# Conduct Bert Emotion analysis on cleaned sentences from Q&Asection (JPM)
from transformers import pipeline
emotion_analyzer = pipeline("text-classification", model="bhadresh-savani/bert-base-uncased-emotion")
documents = (
    jpm_qa_data["cleaned_text"]
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))
    .tolist()
 )
basic_emotion = emotion_analyzer(documents, truncation=True)

print("Sentence emotion analysis:")
print(basic_emotion)

# Add emotion and score to the dataframe
jpm_qa_data["emotion"] = [item["label"] for item in basic_emotion]
jpm_qa_data["score"] = [item["score"] for item in basic_emotion]
jpm_qa_data.head(2)

In [ ]:
# Create dataframe to calculate mean and standard deviation of every emotion every quarter (UBS)
jpm_qa_emotion_stats = jpm_qa_data.groupby(["quarter", "emotion"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std")
).reset_index()
jpm_qa_emotion_stats.head(10)

In [ ]:
# Plot mean and standard deviation of every emotion every quarter as a scatter plot on the same plot (JPM)
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 8))
for emotion in jpm_qa_emotion_stats["emotion"].unique():
    jpm_emotion_qa_data = jpm_qa_emotion_stats[jpm_qa_emotion_stats["emotion"] == emotion]
    # Add lines
    # use pre-defined colours for emotions
    #color = emotion_colors.get(emotion, "black")
    plt.plot(jpm_emotion_qa_data["quarter"], jpm_emotion_qa_data["mean_score"], label=f"{emotion} (average)", marker='o')
plt.title("JPM Q&A Section - Average Emotion Scores Every Quarter")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
# Plot emotions fear and surprise only as grouped barchart (JPM)
import seaborn as sns
plt.figure(figsize=(12, 8))
fear_surprise_qa_data = jpm_qa_emotion_stats[jpm_qa_emotion_stats["emotion"].isin(["fear", "surprise"])]
sns.barplot(data=fear_surprise_qa_data, x="quarter", y="mean_score", hue="emotion")
plt.title("JPM Q&A Section - Average Emotion Scores Every Quarter for Q&A Section (Fear and Surprise)")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.show()

## Main Presenters (JPM)
Create a subset with just the main speakers from the presentation section

In [ ]:
# Filter JPM qa data to just include speaker_name = Jeremy Barnum and Jamie Dimon
jpm_qa_filtered = jpm_qa_data[(jpm_qa_data['speaker_name'] == 'Jeremy Barnum') | (jpm_qa_data['speaker_name'] == 'Jamie Dimon')]
jpm_qa_filtered.head(2)


In [ ]:
# Create dataframe to calculate mean and standard deviation of every emotion every quarter (UBS)
jpm_qa_filtered_emotion_stats = jpm_qa_filtered.groupby(["quarter", "emotion"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std")
).reset_index()
jpm_qa_filtered_emotion_stats.head(10)

In [ ]:
# Plot emotions fear and surprise only as grouped barchart (UBS)
import seaborn as sns
plt.figure(figsize=(12, 8))
jpm_fear_surprise_qa_filtered_data = jpm_qa_filtered_emotion_stats[jpm_qa_filtered_emotion_stats["emotion"].isin(["fear", "surprise"])]
sns.barplot(data=jpm_fear_surprise_qa_filtered_data, x="quarter", y="mean_score", hue="emotion")
plt.title("JPM Q&A Section (Jeremy Barnum & Jamie Dimon) - Average Emotion Scores Every Quarter for Q&A Section (Fear and Surprise)")
plt.xlabel("Quarter")
plt.ylabel("Score")
plt.show()

The above are just the named speakers answering questions - UBS looks the same, but JPM is different, suggesting the sentiment of the questioner has an impact

## Sentiment comparisons - JPM

In [ ]:
#jpm_qa_filtered_emotion_stats
#jpm_prepared_emotion_stats

# Create one dataframe from jpm_prepared_emotion_stats and jpm_qa_filtered_emotion_stats for comparison plots and retain column for prepared vs Q&A and if a value is missing then set to 0
jpm_prepared_emotion_stats["source"] = "prepared"
jpm_qa_filtered_emotion_stats["source"] = "qa_filtered"
jpm_combined_emotion_stats = pd.concat([jpm_prepared_emotion_stats, jpm_qa_filtered_emotion_stats], ignore_index=True)

# If a value is NaN for mean score or std_score, fill it with 0
jpm_combined_emotion_stats["mean_score"] = jpm_combined_emotion_stats["mean_score"].fillna(0)
jpm_combined_emotion_stats["std_score"] = jpm_combined_emotion_stats["std_score"].fillna(0)
jpm_combined_emotion_stats.tail(20)



In [ ]:
all_quarters = sorted(
    set(jpm_prepared_emotion_stats["quarter"]) |
    set(jpm_qa_filtered_emotion_stats["quarter"])
)
all_emotions = sorted(
    set(jpm_prepared_emotion_stats["emotion"]) |
    set(jpm_qa_filtered_emotion_stats["emotion"])
)

full_index = pd.MultiIndex.from_product(
    [all_quarters, all_emotions],
    names=["quarter", "emotion"]
)

prepared_full = (
    jpm_prepared_emotion_stats
    [["quarter", "emotion", "mean_score"]]
    .drop_duplicates()
    .set_index(["quarter", "emotion"])
    .reindex(full_index)
    .reset_index()
    .rename(columns={"mean_score": "prepared"})
)

qa_full = (
    jpm_qa_filtered_emotion_stats
    [["quarter", "emotion", "mean_score"]]
    .drop_duplicates()
    .set_index(["quarter", "emotion"])
    .reindex(full_index)
    .reset_index()
    .rename(columns={"mean_score": "qa_filtered"})
)

jpm_diff_emotion_stats = prepared_full.merge(
    qa_full,
    on=["quarter", "emotion"],
    how="outer"
)

jpm_diff_emotion_stats["diff"] = (
    jpm_diff_emotion_stats["qa_filtered"] -
    jpm_diff_emotion_stats["prepared"]
)

In [ ]:
jpm_diff_emotion_stats.tail(20)

In [ ]:
# Display the combined emotion stats for JPM Q&A and prepared data as a scatter plot


plt.figure(figsize=(10, 6))
sns.barplot(data=jpm_diff_emotion_stats, x="quarter", y="diff", hue="emotion")
plt.axhline(0, linestyle='--', color='gray')
# Add axis line for 20% change in emotion scores
plt.axhline(0.2, linestyle='--', color='red')
plt.axhline(-0.2, linestyle='--', color='red')
plt.title("JPM: Difference in Emotion Scores for Jeremy Barnum & Jamie Dimon (Subtracting Prepared from Q&A)")
plt.show()


